# ScanOps V13 — **7B** QLoRA 학습 (CVEfixes 3,484 · 자체 판별력 강화)

V12(264개)의 학습 부족을 해소: HF `hitoshura25/cvefixes` 코드쌍에서 추출한
**3,484개**(취약1772/안전1712, 10개 언어)로 13배 확장. 누수차단:
held-out 벤치 141 CVE-id 제외 + train/val CVE-disjoint.

하이퍼파라미터: r=32 / alpha=64 / dropout=0.05 (메모리 결정 유지). 어댑터만 저장(병합 X).
데이터가 13배라 학습 시간↑(A100 ~60~90분). val loss 발산 시 EPOCHS↓.

## ① 의존성 + GPU


In [ ]:
!pip -q install transformers peft datasets accelerate bitsandbytes matplotlib
import torch
assert torch.cuda.is_available(), '런타임 → 런타임 유형 변경 → GPU 선택(A100 권장, 없으면 L4). T4는 비권장.'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory/1e9
bf16 = torch.cuda.is_bf16_supported()
print(f'GPU: {name} | VRAM {vram:.0f}GB | bf16지원: {bf16}')
if not bf16:
    print('⚠️ bf16 미지원(T4 등) → fp16 자동전환·느림·OOM위험. A100/L4 권장.')
elif vram < 22:
    print('⚠️ VRAM 작음 — OOM 시 학습셀의 MAXLEN을 640으로 낮추세요.')


## ② 파일 업로드 (3B 때와 같은 두 파일)


In [ ]:
import os
for f in ['lora_train_v13.jsonl', 'lora_train_v13_val.jsonl']:
    if os.path.exists(f): os.remove(f)
from google.colab import files
files.upload()
print('학습:', os.path.exists('lora_train_v13.jsonl'),
      '| 검증:', os.path.exists('lora_train_v13_val.jsonl'))

## ③ 7B QLoRA 학습 (~50~70분, 3 epoch, val loss 감시)


In [ ]:
import torch, gc
try: del trainer, model
except: pass
gc.collect(); torch.cuda.empty_cache()

import json, time, os
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainingArguments)

TRAIN = 'lora_train_v13.jsonl'
VAL   = 'lora_train_v13_val.jsonl'
BASE  = 'Qwen/Qwen2.5-Coder-7B-Instruct'        # ★ 7B
MAXLEN, R, ALPHA, EPOCHS, LR = 768, 32, 64, 3, 1e-4   # v13: 코드가 길어 768(트렁케이션 0.2%). OOM 시 640
SYS = 'You are a security code analyzer.'

def load(p): return [json.loads(l) for l in open(p) if l.strip()]
tr_rows, va_rows = load(TRAIN), load(VAL)
def safe_pct(rows): return 100*sum(1 for r in rows if r['completion'].upper().startswith('VULNERABILITY: NONE'))/len(rows)
print(f'train {len(tr_rows)} (안전 {safe_pct(tr_rows):.1f}%) | val {len(va_rows)}')

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
def fmt(e): return {'text': f'<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\n'+e['prompt']+'<|im_end|>\n<|im_start|>assistant\n'+e['completion']+'<|im_end|>'}
def tk(e):
    o=tok(e['text'],truncation=True,max_length=MAXLEN); o['labels']=o['input_ids'].copy(); return o
tr = Dataset.from_list(tr_rows).map(fmt).map(tk, remove_columns=['prompt','completion','text'])
va = Dataset.from_list(va_rows).map(fmt).map(tk, remove_columns=['prompt','completion','text'])

USE_BF16 = torch.cuda.is_bf16_supported()        # A100/L4=True, T4=False(자동 fp16)
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f'compute_dtype={COMPUTE_DTYPE} (bf16={USE_BF16})')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type=TaskType.CAUSAL_LM))
model.print_trainable_parameters()

args = TrainingArguments(output_dir='out', num_train_epochs=EPOCHS, per_device_train_batch_size=1,
    per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=LR,
    lr_scheduler_type='cosine', warmup_ratio=0.03, fp16=not USE_BF16, bf16=USE_BF16, logging_steps=5,
    eval_strategy='epoch', save_strategy='no', report_to='none',
    gradient_checkpointing=True, optim='paged_adamw_8bit')
trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=va,
    data_collator=DataCollatorForSeq2Seq(tok, pad_to_multiple_of=8, return_tensors='pt'))
t0=time.time(); trainer.train(); print(f'학습 {(time.time()-t0)/60:.1f}분')
model.save_pretrained('adapter'); tok.save_pretrained('adapter')
json.dump([e for e in trainer.state.log_history], open('adapter/train_log.json','w'), indent=2)

## ④ 학습곡선 + 과적합 점검


In [ ]:
import matplotlib.pyplot as plt, json
log = json.load(open('adapter/train_log.json'))
tr_l = [(e['step'], e['loss']) for e in log if 'loss' in e]
va_l = [(e['step'], e['eval_loss']) for e in log if 'eval_loss' in e]
plt.figure(figsize=(7,4))
if tr_l: plt.plot(*zip(*tr_l), label='train loss')
if va_l: plt.plot(*zip(*va_l), 'o-', label='val loss (epoch)')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.title('V13-7B learning curve')
plt.grid(alpha=.3); plt.savefig('adapter/v13_7b_learning_curve.png', dpi=120, bbox_inches='tight'); plt.show()
if va_l:
    print('val loss per epoch:', [round(v,4) for _,v in va_l])
    print('⚠️ 반등 시 과적합' if (len(va_l)>=2 and va_l[-1][1] > min(v for _,v in va_l)+0.05) else '✅ 안정')

## ⑤ 어댑터 다운로드


In [ ]:
!cd adapter && zip -qr ../adapter_v13_7b.zip . && cd ..
from google.colab import files
files.download('adapter_v13_7b.zip')